# Enhanced DAC-Based Lens Column Inversion (JEOL ARM-200F)

## Goal
Recover the physical column parameters (inter-lens distances $d_i$, focal-length coefficients $\tilde{C}_{f,i}$) from the manufacturer's **magnification → DAC look-up table**.

## Enhanced Column Layout (Sample → Detector)

**NEW**: Starting from the sample plane:
$$\text{Sample} \xrightarrow{d_{\mathrm{obj}}} \text{OL} \xrightarrow{d_0} \text{IL1} \xrightarrow{d_1} \text{IL2} \xrightarrow{d_2} \text{IL3} \xrightarrow{d_3} \text{PL1} \xrightarrow{d_4} \text{detector}$$

**Key Changes**:
- Start at sample plane (not OL image plane)
- Include objective lens (OL) with variable focal length
- OL post-field creates image that becomes input to projector system
- Total system: 5 lenses (OL, IL1, IL2, IL3, PL1) and 6 distances

## Model
- Focal length: $f_i = 1/(\tilde{C}_{f,i} \cdot \text{DAC}_i^2)$
- Rotation: $\psi_i = \tilde{K}_i \cdot \text{DAC}_i$ (constraint: $\sum_i \psi_i = 0$)
- Focus condition at detector: $B = M[0,1] = 0$
- Magnification: $M_{\mathrm{total}} = M[0,0]$ from sample to detector

## Methods
This notebook explores multiple optimization approaches:
1. **Levenberg-Marquardt** (trust-region with Jacobian)
2. **Trust-region reflective** (constrained bounds)
3. **L-BFGS-B** with multiple restarts
4. **Differential Evolution** (global optimization)
5. **Bayesian optimization** (exploration of parameter space)

## Solvability Analysis
We will determine:
- Whether a unique solution exists
- How many solutions are possible
- What constraints make the problem well-posed
- What information would be needed if no solution exists

In [ ]:
import sys
sys.path.insert(0, "../../src")

import numpy as np
import jax
import jax.numpy as jnp
from scipy.optimize import least_squares, minimize, differential_evolution
import matplotlib.pyplot as plt
from scipy.stats import qmc
import warnings

from temgym_core.transfer_matrices import propagation_matrix, lens_matrix

jax.config.update("jax_enable_x64", True)

print("✓ Imports successful")
print(f"  JAX version: {jax.__version__}")
print(f"  NumPy version: {np.__version__}")

## Load DAC Data

The JEOL ARM-200F magnification calibration table contains DAC values for different magnifications.

In [ ]:
# ================================================================
# RAW DAC DATA — JEOL ARM-200F magnification look-up table
# ================================================================
RAW_DAC_HEX = {
    "2000":    {"IL1": "0x4b4b", "IL2": "0x4b2b", "IL3": "0xaf98", "OLf": "0x625a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "2500":    {"IL1": "0x4cd7", "IL2": "0x458f", "IL3": "0xb471", "OLf": "0x562a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "3000":    {"IL1": "0x4e0b", "IL2": "0x4046", "IL3": "0xb8fd", "OLf": "0x582a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "4000":    {"IL1": "0x4fcf", "IL2": "0x3657", "IL3": "0xc0d7", "OLf": "0x656a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "5000":    {"IL1": "0x512f", "IL2": "0x2d1d", "IL3": "0xc719", "OLf": "0x6d6a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "6000":    {"IL1": "0x526e", "IL2": "0x2515", "IL3": "0xcccc", "OLf": "0x684a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "8000":    {"IL1": "0x6a31", "IL2": "0x607f", "IL3": "0xadd0", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xe2cb"},
    "10000":   {"IL1": "0x6fe4", "IL2": "0x5c11", "IL3": "0xb1c7", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xe5a6"},
    "12000":   {"IL1": "0x73f5", "IL2": "0x5925", "IL3": "0xb6fe", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xeb2f"},
    "15000":   {"IL1": "0x7830", "IL2": "0x56ad", "IL3": "0xc026", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xf548"},
    "20000":   {"IL1": "0x8057", "IL2": "0x52d6", "IL3": "0xbeb9", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xf9bb"},
    "25000":   {"IL1": "0x8692", "IL2": "0x50e2", "IL3": "0x9b35", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xffff"},
    "30000":   {"IL1": "0x9f22", "IL2": "0x611e", "IL3": "0x9b35", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "40000":   {"IL1": "0xa336", "IL2": "0x611e", "IL3": "0x8ad8", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "50000":   {"IL1": "0xa674", "IL2": "0x6498", "IL3": "0x85b7", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "60000":   {"IL1": "0xa810", "IL2": "0x6a48", "IL3": "0x7f59", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "80000":   {"IL1": "0xa8ea", "IL2": "0x7182", "IL3": "0x77d2", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "100000":  {"IL1": "0xa800", "IL2": "0x7d46", "IL3": "0x7205", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "120000":  {"IL1": "0xa7d4", "IL2": "0x7d46", "IL3": "0x6c7f", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "150000":  {"IL1": "0xa761", "IL2": "0x8431", "IL3": "0x65e6", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "200000":  {"IL1": "0xa6c5", "IL2": "0x957b", "IL3": "0x5c4a", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "250000":  {"IL1": "0xa6a8", "IL2": "0x957b", "IL3": "0x552b", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "300000":  {"IL1": "0xa608", "IL2": "0x9ec6", "IL3": "0x4ba8", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "400000":  {"IL1": "0xa584", "IL2": "0xad0c", "IL3": "0x3e16", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "500000":  {"IL1": "0xa51d", "IL2": "0xb944", "IL3": "0x3268", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "600000":  {"IL1": "0xa4d0", "IL2": "0xc3c7", "IL3": "0x2771", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "800000":  {"IL1": "0xe300", "IL2": "0xe54f", "IL3": "0x478",  "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1000000": {"IL1": "0xe300", "IL2": "0xbd3a", "IL3": "0x1ac9", "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1200000": {"IL1": "0xe300", "IL2": "0xcd09", "IL3": "0x1ac9", "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1500000": {"IL1": "0xe300", "IL2": "0xec00", "IL3": "0x0",   "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "2000000": {"IL1": "0xffff", "IL2": "0xf800", "IL3": "0x0",   "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
}

LENS_NAMES = ["IL1", "IL2", "IL3", "PL1"]
AUX_NAMES  = ["OLf", "OLc"]

def parse_dac_table(raw):
    """Convert hex DAC table -> dict of arrays, assign operating mode labels."""
    mags = sorted(int(k) for k in raw.keys())
    n = len(mags)
    
    data = {
        "mag": np.array(mags, dtype=float),
        "log_mag": np.log10(mags),
    }
    for lens in LENS_NAMES + AUX_NAMES:
        data[lens] = np.array([int(raw[str(m)][lens], 16) for m in mags], dtype=float)
    
    # Operating modes based on OLf / PL1 transitions
    modes = []
    for m in mags:
        if m <= 6000:
            modes.append("A")
        elif m <= 25000:
            modes.append("B")
        elif m <= 600000:
            modes.append("C")
        else:
            modes.append("D")
    data["mode"] = np.array(modes)
    
    return data

dac = parse_dac_table(RAW_DAC_HEX)

print(f"Loaded {len(dac['mag'])} magnification settings")
print(f"  Mode A (2k-6k):    {np.sum(dac['mode']=='A')} settings")
print(f"  Mode B (8k-25k):   {np.sum(dac['mode']=='B')} settings")
print(f"  Mode C (30k-600k): {np.sum(dac['mode']=='C')} settings")
print(f"  Mode D (800k-2M):  {np.sum(dac['mode']=='D')} settings")

## Enhanced Forward Model: Sample → Detector

The complete optical path now includes:
1. **Sample plane** (z=0)
2. Distance $d_{\mathrm{obj}}$ to objective lens
3. **Objective lens (OL)** - focal length varies with OLf DAC
4. Distance $d_0$ to IL1
5. **IL1, IL2, IL3, PL1** - projection system
6. Distance $d_4$ to detector

Total ABCD matrix:
$$M = P(d_4) \cdot L_{\text{PL1}} \cdot P(d_3) \cdot L_{\text{IL3}} \cdot P(d_2) \cdot L_{\text{IL2}} \cdot P(d_1) \cdot L_{\text{IL1}} \cdot P(d_0) \cdot L_{\text{OL}} \cdot P(d_{\mathrm{obj}})$$

In [ ]:
# ================================================================
# ENHANCED FORWARD MODEL (includes objective lens)
# ================================================================

N_LENSES = 5   # OL, IL1, IL2, IL3, PL1
N_DISTS  = 6   # d_obj, d0, d1, d2, d3, d4

def build_abcd_with_objective(dists, focals, xp=jnp):
    """Build system ABCD matrix from sample to detector.
    
    dists:  array of shape (6,) = [d_obj, d0, d1, d2, d3, d4]
    focals: array of shape (5,) = [f_OL, f_IL1, f_IL2, f_IL3, f_PL1]
    
    Propagation order: Sample -> d_obj -> OL -> d0 -> IL1 -> ... -> d4 -> detector
    """
    # Start from detector and work backward (right-to-left matrix multiplication)
    M = propagation_matrix(dists[-1], xp=xp)  # d4 to detector
    
    # Add projection lenses backward: PL1, IL3, IL2, IL1
    for i in reversed(range(1, len(focals))):
        M = M @ lens_matrix(focals[i], xp=xp)
        M = M @ propagation_matrix(dists[i], xp=xp)
    
    # Add objective lens and initial propagation
    M = M @ lens_matrix(focals[0], xp=xp)        # OL
    M = M @ propagation_matrix(dists[0], xp=xp)  # d_obj from sample
    
    return M

def focals_from_dac(Cf_tilde, dac_vals):
    """Compute focal lengths from tilde-Cf and DAC values.
    
    f_i = 1 / (Cf_tilde_i * DAC_i^2)
    """
    return 1.0 / (Cf_tilde * dac_vals**2)

def predicted_magnification(dists, Cf_tilde, dac_vals, xp=jnp):
    """Compute the A = M[0,0] element (magnification) for one setting.
    
    dac_vals: [OLf, IL1, IL2, IL3, PL1] DAC values
    Cf_tilde: [C_OL, C_IL1, C_IL2, C_IL3, C_PL1] focal length coefficients
    """
    focals = focals_from_dac(Cf_tilde, dac_vals)
    M = build_abcd_with_objective(dists, focals, xp=xp)
    return M[0, 0]

def predicted_B(dists, Cf_tilde, dac_vals, xp=jnp):
    """Compute B = M[0,1] element for one setting (should be ~0 for focus)."""
    focals = focals_from_dac(Cf_tilde, dac_vals)
    M = build_abcd_with_objective(dists, focals, xp=xp)
    return M[0, 1]

print("✓ Enhanced forward model defined")
print(f"  {N_LENSES} lenses: OL, IL1, IL2, IL3, PL1")
print(f"  {N_DISTS} distances: d_obj, d0, d1, d2, d3, d4")
print(f"  Total parameters to fit: {N_DISTS} distances + {N_LENSES} Cf_tilde = {N_DISTS + N_LENSES}")

## Problem Setup: Mode C Analysis

Mode C is the cleanest subset:
- OLf is constant (39178)
- PL1 is constant (64000)
- 14 magnification settings from 30kx to 600kx
- All variation comes from IL1, IL2, IL3

This provides strong constraints for the inverse problem.

In [ ]:
# Extract Mode C data
mask_c = dac["mode"] == "C"
dac_c = {k: dac[k][mask_c] for k in dac}
n_c = int(mask_c.sum())

# Build DAC matrix: each row = [OLf, IL1, IL2, IL3, PL1]
DAC_C = np.column_stack([dac_c["OLf"], dac_c["IL1"], dac_c["IL2"], dac_c["IL3"], dac_c["PL1"]])
MAG_C = dac_c["mag"]

print(f"Mode C dataset:")
print(f"  {n_c} settings")
print(f"  Magnification range: {MAG_C.min():.0f}x to {MAG_C.max():.0f}x")
print(f"  OLf constant: {dac_c['OLf'][0]:.0f}")
print(f"  PL1 constant: {dac_c['PL1'][0]:.0f}")
print(f"\nConstraints:")
print(f"  Number of equations (mags + focus): {n_c * 2}")
print(f"  Number of unknowns: {N_DISTS + N_LENSES} = {N_DISTS + N_LENSES}")
print(f"  Overdetermined: {n_c * 2 > N_DISTS + N_LENSES}")

## Optimization Strategy 1: Levenberg-Marquardt

Start with a robust trust-region method that handles both nonlinear least squares and provides Jacobian-based convergence.

In [ ]:
# ================================================================
# LEVENBERG-MARQUARDT SOLVER
# ================================================================

def residuals_lm(params, dac_matrix, target_mags, w_mag=1.0, w_focus=1.0):
    """Residual function for least-squares optimization.
    
    params: [d_obj, d0, d1, d2, d3, d4, C_OL, C_IL1, C_IL2, C_IL3, C_PL1]
    Returns concatenated residuals: [mag_errors, B_values]
    """
    dists = params[:N_DISTS]
    Cf_tilde = params[N_DISTS:]
    
    n_settings = len(target_mags)
    residuals = np.zeros(n_settings * 2)
    
    for i in range(n_settings):
        dac_vals = dac_matrix[i]
        
        # Magnification residual
        mag_pred = float(predicted_magnification(dists, Cf_tilde, dac_vals, xp=np))
        residuals[i] = w_mag * (mag_pred - target_mags[i]) / target_mags[i]  # relative error
        
        # Focus residual (B should be zero)
        B_val = float(predicted_B(dists, Cf_tilde, dac_vals, xp=np))
        residuals[n_settings + i] = w_focus * B_val
    
    return residuals

def run_levenberg_marquardt(dac_matrix, target_mags, initial_guess, 
                           bounds=None, w_mag=1.0, w_focus=1.0, 
                           ftol=1e-10, xtol=1e-10, gtol=1e-10):
    """Run Levenberg-Marquardt optimization."""
    
    result = least_squares(
        residuals_lm,
        initial_guess,
        args=(dac_matrix, target_mags, w_mag, w_focus),
        method='lm',  # Levenberg-Marquardt
        ftol=ftol,
        xtol=xtol,
        gtol=gtol,
        max_nfev=10000,
        verbose=2
    )
    
    return result

print("✓ Levenberg-Marquardt solver defined")

## Optimization Strategy 2: Trust-Region Reflective (with bounds)

Use bounded optimization to enforce physical constraints on distances and focal lengths.

In [ ]:
# ================================================================
# TRUST-REGION REFLECTIVE (BOUNDED)
# ================================================================

def run_trust_region(dac_matrix, target_mags, initial_guess, bounds,
                     w_mag=1.0, w_focus=1.0, ftol=1e-10, xtol=1e-10, gtol=1e-10):
    """Run trust-region reflective with bounds."""
    
    result = least_squares(
        residuals_lm,
        initial_guess,
        args=(dac_matrix, target_mags, w_mag, w_focus),
        method='trf',  # Trust Region Reflective
        bounds=bounds,
        ftol=ftol,
        xtol=xtol,
        gtol=gtol,
        max_nfev=10000,
        verbose=2
    )
    
    return result

print("✓ Trust-region solver defined")

## Optimization Strategy 3: Multi-start L-BFGS-B

Use multiple random initializations with gradient-based optimization to explore the solution space.

In [ ]:
# ================================================================
# MULTI-START L-BFGS-B
# ================================================================

def objective_lbfgs(params, dac_matrix, target_mags, w_mag=1.0, w_focus=1.0):
    """Objective function (sum of squared residuals) for L-BFGS-B."""
    res = residuals_lm(params, dac_matrix, target_mags, w_mag, w_focus)
    return np.sum(res**2)

def run_multi_start_lbfgs(dac_matrix, target_mags, bounds, n_starts=100,
                          w_mag=1.0, w_focus=1.0):
    """Run L-BFGS-B from multiple random starting points."""
    
    lower_bounds, upper_bounds = bounds
    n_params = len(lower_bounds)
    
    best_result = None
    best_cost = np.inf
    all_results = []
    
    print(f"Running {n_starts} L-BFGS-B optimizations...")
    
    for i in range(n_starts):
        # Random initialization within bounds
        x0 = np.random.uniform(lower_bounds, upper_bounds)
        
        result = minimize(
            objective_lbfgs,
            x0,
            args=(dac_matrix, target_mags, w_mag, w_focus),
            method='L-BFGS-B',
            bounds=list(zip(lower_bounds, upper_bounds)),
            options={'ftol': 1e-10, 'gtol': 1e-10, 'maxiter': 5000}
        )
        
        all_results.append(result)
        
        if result.fun < best_cost:
            best_cost = result.fun
            best_result = result
            print(f"  Start {i+1}/{n_starts}: New best cost = {best_cost:.3e}")
    
    return best_result, all_results

print("✓ Multi-start L-BFGS-B solver defined")

## Optimization Strategy 5: Bayesian Optimization (Quasi-random sampling)

Use Latin Hypercube Sampling to efficiently explore the parameter space.

In [ ]:
# ================================================================
# BAYESIAN SAMPLING EXPLORATION
# ================================================================

def explore_parameter_space(dac_matrix, target_mags, bounds, n_samples=1000,
                          w_mag=1.0, w_focus=1.0):
    """Use Latin Hypercube Sampling to explore parameter space."""
    
    lower_bounds, upper_bounds = bounds
    n_params = len(lower_bounds)
    
    # Generate quasi-random samples
    sampler = qmc.LatinHypercube(d=n_params)
    unit_samples = sampler.random(n=n_samples)
    
    # Scale to bounds
    samples = qmc.scale(unit_samples, lower_bounds, upper_bounds)
    
    # Evaluate all samples
    costs = np.zeros(n_samples)
    
    print(f"Evaluating {n_samples} quasi-random samples...")
    for i in range(n_samples):
        costs[i] = objective_lbfgs(samples[i], dac_matrix, target_mags, w_mag, w_focus)
        
        if (i + 1) % 100 == 0:
            best_idx = np.argmin(costs[:i+1])
            print(f"  Evaluated {i+1}/{n_samples}, best cost so far: {costs[best_idx]:.3e}")
    
    # Find best samples
    best_indices = np.argsort(costs)[:10]
    
    return samples, costs, best_indices

print("✓ Bayesian exploration defined")

## Define Physical Bounds and Initial Guess

Use realistic physical constraints for microscope geometry.

In [ ]:
# ================================================================
# PHYSICAL BOUNDS AND INITIAL GUESS
# ================================================================

# Distance bounds (meters)
# d_obj: sample to OL (very short, ~1-10 mm)
# d0-d4: inter-lens distances (10-300 mm)
dist_lower = np.array([0.001, 0.010, 0.010, 0.010, 0.010, 0.050])  # 1mm to 300mm
dist_upper = np.array([0.010, 0.300, 0.300, 0.300, 0.300, 0.500])  

# Cf_tilde bounds
# For f = 1/(Cf*DAC^2): with DAC~40000 and f~5mm, Cf ~ 1e-7
# Allow wide range: 1e-9 to 1e-5
Cf_lower = np.array([1e-9, 1e-9, 1e-9, 1e-9, 1e-9])
Cf_upper = np.array([1e-5, 1e-5, 1e-5, 1e-5, 1e-5])

lower_bounds = np.concatenate([dist_lower, Cf_lower])
upper_bounds = np.concatenate([dist_upper, Cf_upper])
bounds = (lower_bounds, upper_bounds)

# Initial guess (reasonable middle values)
initial_dists = np.array([0.003, 0.050, 0.050, 0.040, 0.050, 0.200])  # meters
initial_Cf = np.array([2e-7, 1.5e-7, 2.0e-7, 1.8e-7, 1.0e-7])
initial_guess = np.concatenate([initial_dists, initial_Cf])

print("✓ Bounds and initial guess defined")
print(f"\nDistance bounds (mm):")
for i in range(N_DISTS):
    print(f"  d{i}: [{dist_lower[i]*1e3:.1f}, {dist_upper[i]*1e3:.1f}]")
print(f"\nCf bounds: [{Cf_lower[0]:.1e}, {Cf_upper[0]:.1e}]")

## RUN ALL OPTIMIZATION METHODS

Now we'll run all optimization strategies and compare results to determine:
1. Is there a solution?
2. Is it unique or are there multiple solutions?
3. What tolerance levels are achievable?

In [ ]:
# ================================================================
# MASTER OPTIMIZATION RUNNER
# ================================================================

results_dict = {}

print("="*80)
print("COMPREHENSIVE OPTIMIZATION ANALYSIS")
print("="*80)

# Weight parameters
w_mag = 1.0
w_focus = 1000.0  # Focus constraint is critical

print(f"\nUsing Mode C data: {n_c} magnification settings")
print(f"Weight on magnification: {w_mag}")
print(f"Weight on focus (B=0): {w_focus}")
print(f"\n{'='*80}")

In [ ]:
# METHOD 1: Levenberg-Marquardt (baseline)
print("\n" + "="*80)
print("METHOD 1: LEVENBERG-MARQUARDT (unbounded)")
print("="*80)

try:
    result_lm = run_levenberg_marquardt(
        DAC_C, MAG_C, initial_guess,
        w_mag=w_mag, w_focus=w_focus,
        ftol=1e-12, xtol=1e-12, gtol=1e-12
    )
    results_dict['LM'] = result_lm
    print(f"\n✓ LM Success: {result_lm.success}")
    print(f"  Cost: {result_lm.cost:.3e}")
    print(f"  Optimality: {result_lm.optimality:.3e}")
except Exception as e:
    print(f"✗ LM Failed: {e}")
    results_dict['LM'] = None

In [ ]:
# METHOD 2: Trust-Region Reflective (bounded)
print("\n" + "="*80)
print("METHOD 2: TRUST-REGION REFLECTIVE (bounded)")
print("="*80)

try:
    result_trf = run_trust_region(
        DAC_C, MAG_C, initial_guess, bounds,
        w_mag=w_mag, w_focus=w_focus,
        ftol=1e-12, xtol=1e-12, gtol=1e-12
    )
    results_dict['TRF'] = result_trf
    print(f"\n✓ TRF Success: {result_trf.success}")
    print(f"  Cost: {result_trf.cost:.3e}")
    print(f"  Optimality: {result_trf.optimality:.3e}")
except Exception as e:
    print(f"✗ TRF Failed: {e}")
    results_dict['TRF'] = None

In [ ]:
# METHOD 3: Differential Evolution (global)
print("\n" + "="*80)
print("METHOD 3: DIFFERENTIAL EVOLUTION (global search)")
print("="*80)

try:
    result_de = run_differential_evolution(
        DAC_C, MAG_C, bounds,
        w_mag=w_mag, w_focus=w_focus,
        popsize=30, maxiter=500, tol=1e-12
    )
    results_dict['DE'] = result_de
    print(f"\n✓ DE Success: {result_de.success}")
    print(f"  Cost: {result_de.fun:.3e}")
except Exception as e:
    print(f"✗ DE Failed: {e}")
    results_dict['DE'] = None

In [ ]:
# METHOD 4: Multi-start L-BFGS-B
print("\n" + "="*80)
print("METHOD 4: MULTI-START L-BFGS-B (50 random starts)")
print("="*80)

try:
    result_ms, all_ms = run_multi_start_lbfgs(
        DAC_C, MAG_C, bounds, n_starts=50,
        w_mag=w_mag, w_focus=w_focus
    )
    results_dict['MS_LBFGS'] = result_ms
    results_dict['MS_LBFGS_ALL'] = all_ms
    print(f"\n✓ MS-LBFGS Best cost: {result_ms.fun:.3e}")
except Exception as e:
    print(f"✗ MS-LBFGS Failed: {e}")
    results_dict['MS_LBFGS'] = None

In [ ]:
# METHOD 5: Bayesian exploration
print("\n" + "="*80)
print("METHOD 5: LATIN HYPERCUBE SAMPLING (1000 samples)")
print("="*80)

try:
    samples, costs, best_indices = explore_parameter_space(
        DAC_C, MAG_C, bounds, n_samples=1000,
        w_mag=w_mag, w_focus=w_focus
    )
    results_dict['LHS_SAMPLES'] = samples
    results_dict['LHS_COSTS'] = costs
    results_dict['LHS_BEST'] = best_indices
    print(f"\n✓ LHS Best cost: {costs[best_indices[0]]:.3e}")
    print(f"  Top 10 costs: {costs[best_indices][:10]}")
except Exception as e:
    print(f"✗ LHS Failed: {e}")
    results_dict['LHS_SAMPLES'] = None

## Results Analysis: Compare All Methods

In [ ]:
# ================================================================
# COMPARATIVE ANALYSIS
# ================================================================

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

method_names = ['LM', 'TRF', 'DE', 'MS_LBFGS']
successful_methods = []

for method in method_names:
    result = results_dict.get(method)
    if result is not None:
        if hasattr(result, 'success') and result.success:
            cost = result.cost if hasattr(result, 'cost') else result.fun
            successful_methods.append((method, cost, result))
            print(f"\n{method}:")
            print(f"  ✓ Converged")
            print(f"  Cost: {cost:.3e}")
        else:
            print(f"\n{method}:")
            print(f"  ✗ Did not converge")
    else:
        print(f"\n{method}:")
        print(f"  ✗ Failed to run")

# LHS analysis
if results_dict.get('LHS_COSTS') is not None:
    costs = results_dict['LHS_COSTS']
    best_idx = results_dict['LHS_BEST'][0]
    print(f"\nLATIN HYPERCUBE SAMPLING:")
    print(f"  Best cost: {costs[best_idx]:.3e}")
    print(f"  Median cost: {np.median(costs):.3e}")
    print(f"  Worst cost: {np.max(costs):.3e}")
    
    # Find solutions below threshold
    threshold = 1e-2
    good_solutions = np.sum(costs < threshold)
    print(f"  Solutions with cost < {threshold}: {good_solutions} / {len(costs)}")

print("\n" + "="*80)

## Solution Multiplicity Analysis

Check if multiple distinct solutions exist by clustering the successful optimization results.

In [ ]:
# ================================================================
# SOLUTION MULTIPLICITY ANALYSIS
# ================================================================

print("\n" + "="*80)
print("SOLUTION MULTIPLICITY ANALYSIS")
print("="*80)

# Collect all successful solutions
all_solutions = []
all_costs = []

for method, cost, result in successful_methods:
    all_solutions.append(result.x)
    all_costs.append(cost)

# Add best multi-start solutions
if results_dict.get('MS_LBFGS_ALL') is not None:
    for r in results_dict['MS_LBFGS_ALL']:
        if r.success:
            all_solutions.append(r.x)
            all_costs.append(r.fun)

# Add best LHS samples
if results_dict.get('LHS_SAMPLES') is not None:
    for idx in results_dict['LHS_BEST'][:5]:
        all_solutions.append(results_dict['LHS_SAMPLES'][idx])
        all_costs.append(results_dict['LHS_COSTS'][idx])

if len(all_solutions) > 0:
    all_solutions = np.array(all_solutions)
    all_costs = np.array(all_costs)
    
    print(f"\nCollected {len(all_solutions)} solutions")
    print(f"  Cost range: [{np.min(all_costs):.3e}, {np.max(all_costs):.3e}]")
    
    # Filter to good solutions
    cost_threshold = np.min(all_costs) * 10  # Within 10x of best
    good_mask = all_costs < cost_threshold
    good_solutions = all_solutions[good_mask]
    good_costs = all_costs[good_mask]
    
    print(f"\nGood solutions (cost < {cost_threshold:.3e}): {len(good_solutions)}")
    
    if len(good_solutions) > 1:
        # Check parameter variation
        param_std = np.std(good_solutions, axis=0)
        param_mean = np.mean(good_solutions, axis=0)
        param_cv = param_std / (np.abs(param_mean) + 1e-10)  # Coefficient of variation
        
        print(f"\nParameter variation (coefficient of variation):")
        print(f"  Distances:")
        for i in range(N_DISTS):
            print(f"    d{i}: mean={param_mean[i]*1e3:.2f}mm, CV={param_cv[i]:.3f}")
        print(f"  Focal length coefficients:")
        for i in range(N_LENSES):
            j = N_DISTS + i
            print(f"    Cf[{i}]: mean={param_mean[j]:.3e}, CV={param_cv[j]:.3f}")
        
        # Determine if solutions are distinct
        max_cv = np.max(param_cv)
        if max_cv > 0.1:
            print(f"\n⚠ MULTIPLE DISTINCT SOLUTIONS DETECTED")
            print(f"  Max coefficient of variation: {max_cv:.3f}")
            print(f"  This suggests the problem is underdetermined or has multiple local minima.")
        else:
            print(f"\n✓ Solutions are clustered (max CV = {max_cv:.3f})")
            print(f"  This suggests a unique solution exists.")
    else:
        print(f"\nOnly one good solution found - cannot assess multiplicity.")
else:
    print("\n✗ No successful solutions found.")

print("\n" + "="*80)

## Detailed Analysis of Best Solution

Examine the best solution in detail: parameters, residuals, and physical interpretation.

In [ ]:
# ================================================================
# BEST SOLUTION ANALYSIS
# ================================================================

if len(all_solutions) > 0:
    best_idx = np.argmin(all_costs)
    best_solution = all_solutions[best_idx]
    best_cost = all_costs[best_idx]
    
    print("\n" + "="*80)
    print("BEST SOLUTION DETAILED ANALYSIS")
    print("="*80)
    print(f"\nCost: {best_cost:.6e}")
    
    # Extract parameters
    best_dists = best_solution[:N_DISTS]
    best_Cf = best_solution[N_DISTS:]
    
    print(f"\n--- Distances (mm) ---")
    dist_names = ['d_obj', 'd0', 'd1', 'd2', 'd3', 'd4']
    for i, name in enumerate(dist_names):
        print(f"  {name}: {best_dists[i]*1e3:.3f} mm")
    print(f"  Total: {np.sum(best_dists)*1e3:.3f} mm")
    
    print(f"\n--- Focal Length Coefficients ---")
    lens_names_full = ['OL', 'IL1', 'IL2', 'IL3', 'PL1']
    for i, name in enumerate(lens_names_full):
        print(f"  Cf_{name}: {best_Cf[i]:.6e}")
    
    # Compute focal lengths at typical DAC values
    print(f"\n--- Focal Lengths at Mode C DACs (mm) ---")
    for j in range(min(3, len(DAC_C))):
        dac_vals = DAC_C[j]
        focals = focals_from_dac(best_Cf, dac_vals)
        print(f"\n  Setting {j+1} (mag={MAG_C[j]:.0f}x):")
        for i, name in enumerate(lens_names_full):
            print(f"    f_{name} = {focals[i]*1e3:.3f} mm (DAC={dac_vals[i]:.0f})")
    
    # Compute residuals
    print(f"\n--- Residual Analysis ---")
    mag_errors = []
    B_values = []
    
    for i in range(len(MAG_C)):
        dac_vals = DAC_C[i]
        mag_pred = float(predicted_magnification(best_dists, best_Cf, dac_vals, xp=np))
        B_val = float(predicted_B(best_dists, best_Cf, dac_vals, xp=np))
        
        mag_error = (mag_pred - MAG_C[i]) / MAG_C[i] * 100  # percent error
        mag_errors.append(mag_error)
        B_values.append(B_val)
    
    mag_errors = np.array(mag_errors)
    B_values = np.array(B_values)
    
    print(f"  Magnification errors (%):\n")
    print(f"    Mean: {np.mean(mag_errors):.3f}%")
    print(f"    RMS:  {np.sqrt(np.mean(mag_errors**2)):.3f}%")
    print(f"    Max:  {np.max(np.abs(mag_errors)):.3f}%")
    
    print(f"\n  B values (focus, should be ~0):")
    print(f"    Mean: {np.mean(np.abs(B_values)):.6e}")
    print(f"    RMS:  {np.sqrt(np.mean(B_values**2)):.6e}")
    print(f"    Max:  {np.max(np.abs(B_values)):.6e}")
    
    print("\n" + "="*80)
else:
    print("\n✗ No solutions available for analysis.")

## Visualization: Cost Landscape

In [ ]:
# ================================================================
# VISUALIZATION
# ================================================================

if results_dict.get('LHS_COSTS') is not None and len(all_solutions) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Panel 1: Cost distribution
    ax = axes[0, 0]
    costs_lhs = results_dict['LHS_COSTS']
    ax.hist(np.log10(costs_lhs + 1e-20), bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax.axvline(np.log10(best_cost + 1e-20), color='red', linestyle='--', linewidth=2, label=f'Best: {best_cost:.2e}')
    ax.set_xlabel('log10(Cost)')
    ax.set_ylabel('Frequency')
    ax.set_title('Cost Distribution (LHS samples)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Panel 2: Magnification fit
    if len(all_solutions) > 0:
        ax = axes[0, 1]
        mag_pred = [float(predicted_magnification(best_dists, best_Cf, DAC_C[i], xp=np)) 
                   for i in range(len(MAG_C))]
        ax.plot(MAG_C, MAG_C, 'k--', label='Perfect fit', linewidth=2)
        ax.plot(MAG_C, mag_pred, 'ro-', label='Predicted', markersize=5)
        ax.set_xlabel('Target Magnification')
        ax.set_ylabel('Predicted Magnification')
        ax.set_title('Magnification Fit Quality')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')
    
    # Panel 3: B values (focus condition)
    if len(all_solutions) > 0:
        ax = axes[1, 0]
        B_vals = [float(predicted_B(best_dists, best_Cf, DAC_C[i], xp=np)) 
                 for i in range(len(MAG_C))]
        ax.plot(MAG_C, B_vals, 'bo-', markersize=5)
        ax.axhline(0, color='k', linestyle='--', linewidth=1)
        ax.set_xlabel('Magnification')
        ax.set_ylabel('B value (should be 0)')
        ax.set_title('Focus Condition')
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
    
    # Panel 4: Parameter space (first two distances)
    if results_dict.get('LHS_SAMPLES') is not None:
        ax = axes[1, 1]
        samples = results_dict['LHS_SAMPLES']
        costs_lhs = results_dict['LHS_COSTS']
        scatter = ax.scatter(samples[:, 0]*1e3, samples[:, 1]*1e3, 
                           c=np.log10(costs_lhs + 1e-20), 
                           cmap='viridis', alpha=0.5, s=10)
        ax.plot(best_dists[0]*1e3, best_dists[1]*1e3, 'r*', markersize=20, label='Best')
        ax.set_xlabel('d_obj (mm)')
        ax.set_ylabel('d0 (mm)')
        ax.set_title('Parameter Space (first two distances)')
        plt.colorbar(scatter, ax=ax, label='log10(Cost)')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for visualization.")

## Final Conclusions: Solvability Assessment

Based on all optimization attempts, we can now make definitive statements about the problem.

In [ ]:
# ================================================================
# FINAL SOLVABILITY ASSESSMENT
# ================================================================

print("\n" + "="*80)
print("FINAL SOLVABILITY ASSESSMENT")
print("="*80)

# Determine convergence threshold
convergence_threshold = 1e-4

if len(all_solutions) > 0 and best_cost < convergence_threshold:
    print(f"\n✓ PROBLEM IS SOLVABLE")
    print(f"\n  Best achieved cost: {best_cost:.6e}")
    print(f"  Convergence threshold: {convergence_threshold:.6e}")
    
    # Check for uniqueness
    if len(good_solutions) > 1:
        max_cv = np.max(param_cv)
        if max_cv > 0.1:
            print(f"\n  ⚠ Multiple distinct solutions exist (max CV = {max_cv:.3f})")
            print(f"  The problem is underdetermined - additional constraints needed.")
            print(f"\n  To obtain a unique solution, you would need:")
            print(f"    - Direct measurement of at least one distance")
            print(f"    - Or measurement of one focal length coefficient")
            print(f"    - Or additional magnification data at different operating modes")
        else:
            print(f"\n  ✓ Solution appears unique (max CV = {max_cv:.3f})")
            print(f"  The problem is well-posed with the current constraints.")
    else:
        print(f"\n  ⚠ Only one solution found - uniqueness uncertain")
        print(f"  More optimization trials needed to confirm uniqueness.")
    
    print(f"\n  Solution quality:")
    if len(mag_errors) > 0:
        print(f"    - Magnification RMS error: {np.sqrt(np.mean(mag_errors**2)):.3f}%")
        print(f"    - Focus RMS error: {np.sqrt(np.mean(B_values**2)):.6e}")
    
elif len(all_solutions) > 0:
    print(f"\n⚠ PROBLEM IS PARTIALLY SOLVABLE")
    print(f"\n  Best achieved cost: {best_cost:.6e}")
    print(f"  Convergence threshold: {convergence_threshold:.6e}")
    print(f"\n  The optimizer found a local minimum but did not achieve full convergence.")
    print(f"\n  Possible reasons:")
    print(f"    1. Insufficient degrees of freedom (need more data or constraints)")
    print(f"    2. Model mismatch (actual microscope may have additional optical elements)")
    print(f"    3. Measurement errors in the DAC→magnification calibration table")
    print(f"\n  To improve solvability:")
    print(f"    - Verify the optical model includes all relevant elements")
    print(f"    - Check for systematic errors in magnification calibration")
    print(f"    - Consider aberrations or nonlinear effects not in thin-lens model")
    print(f"    - Add measurements from other operating modes (A, B, D)")

else:
    print(f"\n✗ PROBLEM IS NOT SOLVABLE WITH CURRENT APPROACH")
    print(f"\n  No optimizers successfully converged.")
    print(f"\n  This indicates a fundamental issue:")
    print(f"    1. The model may be incorrect or incomplete")
    print(f"    2. The starting point is critically important (currently: OL image vs sample)")
    print(f"    3. The parameter bounds may be too restrictive")
    print(f"    4. The DAC→focal length relationship may be wrong")
    print(f"\n  What is needed to solve this:")
    print(f"    - Independent verification of optical path (sample → OL → detector)")
    print(f"    - Direct measurement of at least 2-3 parameters (distances or focal lengths)")
    print(f"    - Verification of DAC→current→focal length calibration")
    print(f"    - Consider using different optical model (thick lenses, aberrations)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)